In [1]:
RANDOM_STATE = 42

from pathlib import Path
import sys

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

sys.path.insert(0, str(BASE_DIR))

# 03 — Modelagem

## Objetivo

Esta etapa compara modelos de famílias distintas para a classificação binária. A avaliação principal é feita por validação cruzada estratificada no conjunto de treino, mantendo o teste separado para a etapa seguinte.

In [2]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.config import PROCESSED_DATA_DIR, METRICS_DIR, RANDOM_STATE
from src.models import build_models

X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv")["high_quality"]
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv")["high_quality"]

models = build_models(RANDOM_STATE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc"}

## 1. Modelos avaliados

### Logistic Regression

É utilizada como baseline linear e oferece uma referência interpretável para o problema.

### Random Forest

É um conjunto de árvores de decisão capaz de representar relações não lineares e interações entre variáveis.

### Gradient Boosting

É um modelo de boosting que constrói árvores sequencialmente para reduzir erros residuais.

O uso de três abordagens permite comparar um modelo linear com dois métodos não lineares.

In [3]:
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    rows.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean(),
    })

cv_results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
display(cv_results.round(4))
cv_results.to_csv(METRICS_DIR / "model_comparison_cv.csv", index=False)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Random Forest,0.9026,0.7734,0.4492,0.5609,0.9227
1,Gradient Boosting,0.8884,0.6345,0.4738,0.5401,0.9052
2,Logistic Regression,0.7845,0.3719,0.7975,0.5054,0.8694


## 2. Estratégia de seleção

A classe High Quality é minoritária. Por isso, o critério principal é o F1-score, complementado por Recall, Precision e ROC-AUC. Accuracy é apresentada como métrica complementar.

A seleção é baseada na validação cruzada, evitando escolher um modelo apenas pelo desempenho em uma única divisão de teste.

In [4]:
selected_model_name = cv_results.iloc[0]["Model"]
selected_model = models[selected_model_name]
selected_model.fit(X_train, y_train)
print("Selected model:", selected_model_name)

Selected model: Random Forest


### Resultado da seleção

O modelo na primeira posição da tabela de validação cruzada é utilizado como modelo final. O conjunto de teste não participa dessa seleção.